In [2]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/opt/tiger/samantha


# Dataset

In [3]:
# from samantha.dataio.data_bucket import data_bucket
# from recipes.bigmusic.datasets.inference import inference_dataset_from_prompt
# from recipes.bigmusic.datasets.lyrics import LyricsDataModule

# batch_size = 8
# prompt_path = data_bucket(
#     "data/prompts/bigmusic_text_prompts/vocal_prompts_20231018_mixed135.csv",
#     cache=True,
# )

# datamodule = LyricsDataModule.from_dataset_type(
#     dataset_types="mcc60m_vocalB_style_mixed_text_audio", # mcc60m_vocalB_style_mixed_text_audio
#     batch_size=batch_size,
#     sample_rate=24000,
#     sample_duration=[15, 20, 25, 30],
#     shuffle_buffer_size=0, # TODO why?
#     lyrics_max_seq_len=400,
#     num_workers=8,
#     pin_memory=True,
#     resampled=True,
#     shardshuffle=True,
#     max_num_segments=10,
# )

# dataloader = datamodule.train_dataloader()

In [4]:
# from IPython.display import display, Audio
# batch = next(iter(dataloader))
# batch["target_audio"] = batch["target_audio"].to("cuda")

# Audio(batch["target_audio"][0].cpu(), rate=24000)

# MIR Model

In [5]:
# Download model from HDFS

In [6]:
# convert

import torch
def convert_ckpt(ckpt_path: str):
    print("Converting", ckpt_path)
    state_dict = torch.load(ckpt_path)
    new_state_dict = {}
    for k, v in state_dict["state_dict"].items():
        if "audio_tokenizer" not in k:
            new_state_dict[k] = v
    state_dict["state_dict"] = new_state_dict
    torch.save(state_dict, ckpt_path)

In [7]:
convert_ckpt("/mnt/bn/janne-research-xl/models/m1/m1_music_sft_vocal_v0_umm_1layer_classification_vocalgender_step=0030000.ckpt")
convert_ckpt("/mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_mcc30k_genre_zh_step=0030000.ckpt")
convert_ckpt("/mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_scene_zh_step=0030000.ckpt")
convert_ckpt("/mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_mood_zh_step=0030000.ckpt")

Converting /mnt/bn/janne-research-xl/models/m1/m1_music_sft_vocal_v0_umm_1layer_classification_vocalgender_step=0030000.ckpt
2024-01-24 02:55:39,217 - databus.databus_cache - INFO - databus python cache flush thread begin
No module named 'flash_attn'


/opt/tiger/samantha/samantha/dataio/parquet/writer.py:17: UserWarning: simplejson not installed, potential NaN issue in IndexShardWriter.
  warnings.warn("simplejson not installed, potential NaN issue in IndexShardWriter.")


Converting /mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_mcc30k_genre_zh_step=0030000.ckpt
Converting /mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_scene_zh_step=0030000.ckpt
Converting /mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_mood_zh_step=0030000.ckpt


## M1 Models

In [8]:
# load M1 model
from samantha.utils.hdfs_tools import hdfs_get
from recipes.mi1.models.music_sft import MI1_MusicClassificationMusicSFT



model_mcc30k_genre = MI1_MusicClassificationMusicSFT.load_from_checkpoint("/mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_mcc30k_genre_zh_step=0030000.ckpt")
model_mcc30k_genre = model_mcc30k_genre.to("cuda")
print("Vocab:", model_mcc30k_genre.tag_tokenizer.vocab)

model_vocal_gender = MI1_MusicClassificationMusicSFT.load_from_checkpoint("/mnt/bn/janne-research-xl/models/m1/m1_music_sft_vocal_v0_umm_1layer_classification_vocalgender_step=0030000.ckpt")
model_vocal_gender = model_vocal_gender.to("cuda")

print("Vocab:", model_vocal_gender.tag_tokenizer.vocab)


model_mcc30k_mood = MI1_MusicClassificationMusicSFT.load_from_checkpoint("/mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_mood_zh_step=0030000.ckpt")
model_mcc30k_mood = model_mcc30k_mood.to("cuda")

print("Mood:", model_mcc30k_mood.tag_tokenizer.vocab)

model_mcc30k_scene = MI1_MusicClassificationMusicSFT.load_from_checkpoint("/mnt/bn/janne-research-xl/models/m1/m1_music_sft_v0_umm_lr5.0e-5_precision32_1layer_scene_zh_step=0030000.ckpt")
model_mcc30k_scene = model_mcc30k_scene.to("cuda")

print("Scene:", model_mcc30k_mood.tag_tokenizer.vocab)


Vocab: ['Childhood', 'Chinese Style, China-Wave', 'Chinese Style, Chinoiserie Electronic', 'Chinese Style, Chinoiserie Rap', 'Chinese Style, GuFeng Music', 'Chinese Tradition, Chinese Opera', 'Chinese Tradition, Traditional Chinese Folk', 'Classical, Funk', 'Classical, R&B/Soul', 'Devotional', 'Hip Hop, Old School', 'Hip Hop, R&B Rap', 'Hip Hop, Trap Rap', 'Jazz, Jazz Pop', 'Metal', 'Pop, Cantopop', 'Pop, Chinese Pop', 'Pop, Contemporary Pop', 'Pop, Pop Folk', 'Pop, Taiwanese Pop', 'Rock', 'Tuhai, DJ', 'Tuhai, MC', 'Tuhai, VinaHouse']
Vocab: ['chorus', 'female', 'female,chorus', 'female,male', 'male', 'male,chorus']
Mood: ['Calm/Relaxing', 'Cute/Playful', 'Dreamy/Ethereal', 'Dynamic/Energetic', 'Excited', 'Groovy/Funky', 'Happy', 'Healing', 'Inspirational/Hopeful', 'Miss', 'Nostalgic/Memory', 'Romantic', 'Shocking/magnificent/epicl', 'Sorrow/Sad']
Scene: ['Calm/Relaxing', 'Cute/Playful', 'Dreamy/Ethereal', 'Dynamic/Energetic', 'Excited', 'Groovy/Funky', 'Happy', 'Healing', 'Inspiration

/opt/tiger/samantha/recipes/datasets/mir/music_sft.py:35: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


## Audio Tokenizer

In [9]:
# load audio tokenizer
from recipes.mi1.models.tokenizers import UMMTokenizer
audio_tokenizer = UMMTokenizer().to("cuda")

In [10]:
from recipes.datasets.mir.music_sft import MCC30KDataModuleZH, MusicSFTTokenizerGenresZH, MusicSFTMCC30KPreprocessedDataModule

tokenizer = MusicSFTTokenizerGenresZH()
datamodule = MusicSFTMCC30KPreprocessedDataModule(
    tokenizer=tokenizer,
    sample_rate=24000,
    duration=30,
    batch_size=50,
    shuffle_buffer_size=0,
    resampled=False,
    shardshuffle=False,
    num_workers=8
)
train_loader = datamodule.train_dataloader()
batch = next(iter(train_loader))


initializing train datasets...
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Childhood/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Style, China-Wave/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Style, Chinoiserie Electronic/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Style, Chinoiserie Rap/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Style, GuFeng Music/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Tradition, Chinese Opera/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Chinese Tradition, Traditional Chinese Folk/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/music_sft/chinese/genre_1/Classical, Funk/url2index.txt
adding dataset: /mnt/bn/janne-research-xl/data/

In [76]:
## movie
!pip3 install moviepy==1.0.3
# !pip3 install --upgrade decorator==4.4.2

import os
import torchaudio
import matplotlib.pyplot as plt
from moviepy.editor import AudioFileClip, ImageClip

def add_static_image_to_audio(image_path, audio_path, output_path):
    """Create and save a video file to `output_path` after 
    combining a static image that is located in `image_path` 
    with an audio file in `audio_path`"""
    
    # create the audio clip object
    audio_clip = AudioFileClip(audio_path)
    
    # create the image clip object
    image_clip = ImageClip(image_path)
    
    # use set_audio method from image clip to combine the audio with the image
    video_clip = image_clip.set_audio(audio_clip)
    # specify the duration of the new clip to be the duration of the audio clip
    video_clip.duration = audio_clip.duration
    
    # set the FPS to 1
    video_clip.fps = 1
    
    # write the resuling video clip
    video_clip.write_videofile(output_path, fps=1)
    

def predict_output_video(audio):
    
    hidden_states = audio_tokenizer(audio.to("cuda")).hidden_states
    # result = model_vocal_gender.predict_tags(hidden_states)   
    
    for idx in range(len(audio)):
        image_fp = f"output_{idx}.png"
        audio_fp = f"output_{idx}.mp3"
        output_fp = f"{idx}.mp4"
        
        fig, axs = plt.subplots(2, 2, figsize=(25, 15))
        for ax, model, name in zip(axs.flatten(), [model_mcc30k_genre, model_vocal_gender, model_mcc30k_mood, model_mcc30k_scene], ["genre", "vocal_gender", "mood", "scene"]):
            result = model.predict_tags(hidden_states)
            print(batch.tag_names[idx], result.tag_names[idx])

            
            ax.plot(result.tag_probabilities[idx].cpu(), marker="*", color="black")
            ax.set_xticks(range(len(model.tag_tokenizer.vocab)), list(model.tag_tokenizer.vocab), rotation=75)
            ax.set_ylabel("probability")
            ax.set_title(f"{name} - predicted: {result.tag_names[idx]}")
            ax.grid()
        
        plt.tight_layout()
        plt.savefig(image_fp)
        plt.close()

        torchaudio.save(audio_fp, batch.audio[idx].cpu(), 24000)

        add_static_image_to_audio(image_fp, audio_fp, output_fp)
        os.remove(image_fp)
        os.remove(audio_fp)

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://bytedpypi.byted.org/simple/
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [ ]:
predict_output_video(batch.audio)

Tuhai, VinaHouse Tuhai, VinaHouse
Tuhai, VinaHouse female
Tuhai, VinaHouse Dynamic/Energetic
Tuhai, VinaHouse Nightclub
Moviepy - Building video 0.mp4.
MoviePy - Writing audio in 0TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video 0.mp4



Moviepy - Done !
Moviepy - video ready 0.mp4
Pop, Chinese Pop Pop, Chinese Pop
Pop, Chinese Pop female
Pop, Chinese Pop Cute/Playful
Pop, Chinese Pop Graduation
Moviepy - Building video 1.mp4.
MoviePy - Writing audio in 1TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video 1.mp4



Moviepy - Done !
Moviepy - video ready 1.mp4
Hip Hop, R&B Rap Hip Hop, R&B Rap
Hip Hop, R&B Rap female
Hip Hop, R&B Rap Cute/Playful
Hip Hop, R&B Rap Summer
Moviepy - Building video 2.mp4.
MoviePy - Writing audio in 2TEMP_MPY_wvf_snd.mp3


...

Moviepy - Done !
Moviepy - video ready 20.mp4
Pop, Chinese Pop Pop, Chinese Pop
Pop, Chinese Pop female
Pop, Chinese Pop Miss
Pop, Chinese Pop Autumn
Moviepy - Building video 21.mp4.
MoviePy - Writing audio in 21TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video 21.mp4



Moviepy - Done !
Moviepy - video ready 21.mp4
Chinese Style, Chinoiserie Electronic Chinese Style, Chinoiserie Electronic
Chinese Style, Chinoiserie Electronic female
Chinese Style, Chinoiserie Electronic Dreamy/Ethereal
Chinese Style, Chinoiserie Electronic Nightclub
Moviepy - Building video 22.mp4.
MoviePy - Writing audio in 22TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video 22.mp4



Moviepy - Done !
Moviepy - video ready 22.mp4
Hip Hop, Trap Rap Hip Hop, R&B Rap
Hip Hop, Trap Rap female,male
Hip Hop, Trap Rap Dreamy/Ethereal
Hip Hop, Trap Rap Beauty/Fashion
Moviepy - Building video 23.mp4.
MoviePy - Writing audio in 23TEMP_MPY_wvf_snd.mp3


## Full pipeline

In [ ]:
import torch
from recipes.mi1.models.mi1 import MI1, MI1Config
from recipes.mi1.models.mi1 import UMMTokenizer
from samantha.utils.hdfs_tools import hdfs_open, hdfs_get_cache
from samantha.dataio.data_bucket import data_bucket
from recipes.mi1.scripts.inference import DiffusionModels, Q4SemanticModel
from samantha.transforms.tokenizers.phoneme import LyricPhonemeTokenizer

lyric_tokenizer = LyricPhonemeTokenizer()

In [ ]:
from pytorch_lightning import seed_everything
seed_everything(1995)

batch_size = 8
style_text = "A country song with female vocal."
lyrics = "Imagine there's no heaven. It's easy if you try. No hell below us. Above us, only sky. Imagine all the people. Livin' for today. oh oh oh oh oh."

In [ ]:
semantic_model = Q4SemanticModel()


diffusion_models = DiffusionModels()
diffusion_models = diffusion_models.to("cuda")


In [ ]:
from IPython.display import Audio, display
audio = diffusion_models.tokens_to_audio(audio_tokens)
display(Audio(audio[0].cpu(), rate=24000))

In [ ]:
eos_padding_id = 0 # TODO why?
eos_mask = torch.cumsum(audio_tokens == m1_model.eos_token_id, 1) > 0
audio_tokens[eos_mask] = eos_padding_id

In [ ]:
def stereo_diffusion(audio_tokens, seed: int = 42):
    seed_everything(seed)
    pred_audio_left = diffusion_models.tokens_to_audio(audio_tokens)
    seed_everything(seed + 1) # not actually necessary, but just in case we ever set seeds manually in modeling
    pred_audio_right = diffusion_models.tokens_to_audio(audio_tokens)
    pred_stereo = torch.cat((pred_audio_left[:, None, :], pred_audio_right[:, None, :]), dim=1)
    return pred_stereo, pred_audio_left



In [ ]:
from IPython.display import Audio, display

stereo, mono = stereo_diffusion(audio_tokens)
for idx in range(batch_size):
    print("mono")
    display(Audio(mono[idx].cpu(), rate=24000))
    print("stereo")
    display(Audio(stereo[idx].cpu(), rate=24000))

In [ ]:
# from recipes.bigmusic.datasets.transforms.lyrics import LyricsTokenTransform
# lyrics_tokenizer = LyricsTokenTransform.init_espeak_tokenizer(lyrics_max_seq_len=400)

# lyrics = {"lyrics": lyrics}
# lyrics = lyrics_tokenizer(lyrics)

# style_text = [style_text] * batch_size
# lyrics["lyrics"] = [lyrics["lyrics"]] * batch_size
# lyrics["lyrics_normalized_text"] = [lyrics["lyrics_normalized_text"]] * batch_size
# lyrics["lyrics_tokens"] = lyrics["lyrics_tokens"].unsqueeze(dim=0).repeat(batch_size, 1)

# batch = {
#     "style_text": style_text,
#     "lyrics": lyrics["lyrics"],
#     "lyrics_normalized_text": lyrics["lyrics_normalized_text"],
#     "lyrics_tokens": lyrics["lyrics_tokens"],
#     "conditions": "style_text,lyrics_tokens",
# }

# sos_embeds = semantic_model.semantic_module.target_embedder.get_sos_embed(batch_size)
# mulan_emb_inputs = semantic_model.semantic_module.prepare_mulan_inputs(batch, semantic_model.semantic_module.input_embedders["mulan"])
# lyrics_emb_inputs = semantic_model.semantic_module.prepare_lyrics_inputs(batch, semantic_model.semantic_module.input_embedders["lyrics_tokens"])
# cond_embeds = torch.cat((mulan_emb_inputs, lyrics_emb_inputs, sos_embeds), dim=1)
# target_embedder = semantic_model.semantic_module.target_embedder
# # M1
# import torch
# from recipes.mi1.models.mi1 import MI1Input

# tag_names = ["Rock", "Male"]
# inputs = MI1Input(lyrics=lyrics, tag_names=tag_names)
# m1_audio_tokens = m1_model.generate(inputs, precision=torch.bfloat16)
